<a href="https://colab.research.google.com/github/hoangnguyen3101/Application-algorithms/blob/main/Lab_05_Ungdung_Thuattoan_Bai1_Bai5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài thực hành 5 mở rộng: Ứng dụng thuật toán từ Bài 1 đến Bài 5

Trong bài thực hành này, chúng ta tổng hợp các thuật toán đã học từ **Bài 1 đến Bài 5** và áp dụng vào ba nhóm tình huống thực tế:

- **Kinh tế**: chọn danh mục chiến dịch/đầu tư dưới ràng buộc ngân sách.
- **Giao thông vận tải**: tìm tuyến ít chặng nhất, kiểm tra khả năng kết nối và thành phần liên thông.
- **Trí tuệ nhân tạo**: sắp xếp dữ liệu, chọn mẫu gán nhãn và lan truyền nhãn trên đồ thị tương đồng.

Mục tiêu không phải là xây dựng một hệ thống tối ưu hóa hoàn chỉnh, mà là luyện cách **mô hình hóa bài toán**, **chọn thuật toán phù hợp**, **kiểm chứng kết quả**, và **đánh giá độ phức tạp**.

## 1. Mục tiêu

Sau bài thực hành này, sinh viên có thể:

1. Nhận diện vai trò của phân tích độ phức tạp khi dữ liệu tăng kích thước.
2. Cài đặt và so sánh các chiến lược tham lam, chia để trị, quy hoạch động, BFS và DFS.
3. Mô hình hóa một số bài toán kinh tế, giao thông vận tải và AI bằng cấu trúc dữ liệu phù hợp.
4. Đánh giá kết quả thuật toán bằng bảng, biểu đồ và kiểm thử trường hợp biên.
5. Giải thích khi nào một thuật toán đơn giản như greedy/BFS đủ dùng và khi nào cần phương pháp khác.
6. Viết nhận xét ngắn về giới hạn của mô hình thuật toán so với bài toán thực tế.

## 2. Cài đặt môi trường

Notebook chỉ dùng các thư viện phổ biến và có thể chạy trên Google Colab hoặc máy cá nhân. Cell dưới đây cài đặt an toàn bằng subprocess, tránh lỗi thụt lề của lệnh pip.

In [ ]:
import sys
import subprocess

packages = [
    "numpy",
    "matplotlib",
    "pandas",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Cài đặt hoàn tất.")

## 3. Import thư viện và thiết lập seed

Ta cố định seed để dữ liệu mô phỏng có thể tái lập. Điều này quan trọng khi đo thời gian, so sánh thuật toán hoặc viết báo cáo thực nghiệm.

In [ ]:
import random
import time
from collections import deque, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Seed:", SEED)

## 4. Bản đồ thuật toán đã học

| Bài | Nhóm thuật toán | Ý tưởng chính | Ứng dụng trong lab |
|---|---|---|---|
| 1 | Độ phức tạp | Đo chi phí theo kích thước đầu vào | So sánh thời gian chạy thực nghiệm |
| 2 | Tham lam | Chọn tốt nhất cục bộ | Chọn chiến dịch theo tỷ lệ lợi ích/chi phí |
| 3 | Chia để trị | Chia bài toán, giải con, kết hợp | Merge sort và binary search trên dữ liệu kinh tế/AI |
| 4 | Quy hoạch động | Lưu nghiệm bài toán con | Ba lô 0-1 cho ngân sách marketing |
| 5 | Đồ thị, BFS, DFS | Mô hình quan hệ bằng đỉnh/cạnh | Mạng giao thông và đồ thị tương đồng trong AI |

Một câu hỏi xuyên suốt: **mô hình nào làm cho thuật toán đã học trở thành lựa chọn hợp lý?**

## 5. Bộ hàm hỗ trợ chung

Ta viết các hàm nhỏ, thuần Python, dễ kiểm thử và tái sử dụng trong nhiều bối cảnh.

In [ ]:
def time_function(func, *args, repeats=3, **kwargs):
    """Trả về thời gian chạy trung vị của một hàm."""
    times = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        times.append(time.perf_counter() - start)
    return result, float(np.median(times))


def moving_average(values, window=5):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


def print_section(title):
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))

## 6. Ứng dụng kinh tế: chọn chiến dịch dưới ràng buộc ngân sách

Một doanh nghiệp có nhiều chiến dịch marketing. Mỗi chiến dịch có:

- **cost**: chi phí triển khai;
- **benefit**: lợi ích kỳ vọng;
- **risk**: mức rủi ro mô phỏng.

Ta so sánh hai cách chọn:

1. **Tham lam**: sắp xếp theo tỷ lệ $benefit/cost$ rồi chọn đến khi hết ngân sách.
2. **Quy hoạch động ba lô 0-1**: xét mọi khả năng chọn/không chọn để tối đa hóa lợi ích.

Tham lam nhanh và dễ giải thích, nhưng với bài toán 0-1 có thể bỏ lỡ tổ hợp tốt hơn.

In [ ]:
campaigns = pd.DataFrame({
    "campaign": ["A", "B", "C", "D", "E", "F", "G", "H"],
    "cost":     [12,  7,  11,  8,  9,  6,  14,  5],
    "benefit": [24, 13,  23, 15, 16, 12,  28,  9],
    "risk":    [ 3,  2,   4,  3,  2,  1,   5,  1],
})
campaigns["ratio"] = campaigns["benefit"] / campaigns["cost"]
budget = 30
campaigns

In [ ]:
def greedy_campaign_selection(df, budget):
    ordered = df.sort_values("ratio", ascending=False).reset_index(drop=True)
    chosen = []
    total_cost = 0
    total_benefit = 0
    for _, row in ordered.iterrows():
        if total_cost + int(row.cost) <= budget:
            chosen.append(row.campaign)
            total_cost += int(row.cost)
            total_benefit += int(row.benefit)
    return chosen, total_cost, total_benefit


def dp_campaign_selection(df, budget):
    costs = df["cost"].astype(int).to_list()
    benefits = df["benefit"].astype(int).to_list()
    names = df["campaign"].to_list()
    n = len(costs)

    dp = [[0] * (budget + 1) for _ in range(n + 1)]
    take = [[False] * (budget + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        w = costs[i - 1]
        v = benefits[i - 1]
        for c in range(budget + 1):
            dp[i][c] = dp[i - 1][c]
            if w <= c and dp[i - 1][c - w] + v > dp[i][c]:
                dp[i][c] = dp[i - 1][c - w] + v
                take[i][c] = True

    chosen = []
    c = budget
    for i in range(n, 0, -1):
        if take[i][c]:
            chosen.append(names[i - 1])
            c -= costs[i - 1]
    chosen.reverse()
    total_cost = int(df[df["campaign"].isin(chosen)]["cost"].sum())
    return chosen, total_cost, dp[n][budget], dp


greedy_result = greedy_campaign_selection(campaigns, budget)
dp_result = dp_campaign_selection(campaigns, budget)

print("Ngân sách:", budget)
print("Tham lam:", greedy_result)
print("Quy hoạch động:", dp_result[:3])

### Nhận xét

Nếu hai kết quả khác nhau, đây là minh họa quan trọng: **tiêu chí cục bộ tốt chưa chắc tạo nghiệm toàn cục tốt**. Với ngân sách nhỏ và số chiến dịch không quá lớn, quy hoạch động là lựa chọn phù hợp vì đảm bảo tối ưu.

In [ ]:
labels = ["Greedy", "Dynamic programming"]
benefits = [greedy_result[2], dp_result[2]]
costs = [greedy_result[1], dp_result[1]]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(7, 4))
plt.bar(x - width/2, benefits, width, label="Lợi ích")
plt.bar(x + width/2, costs, width, label="Chi phí")
plt.xticks(x, labels)
plt.ylabel("Giá trị")
plt.title("So sánh tham lam và quy hoạch động")
plt.legend()
plt.show()

## 7. Chia để trị trong dữ liệu kinh tế: sắp xếp và tìm kiếm

Trong phân tích kinh tế hoặc vận hành, ta thường cần sắp xếp giao dịch, sản phẩm, khách hàng theo một chỉ số. Ở đây ta dùng **merge sort** để sắp xếp danh sách doanh thu, sau đó dùng **binary search** để tìm vị trí đầu tiên có doanh thu không nhỏ hơn một ngưỡng.

Đây là một quy trình phổ biến:

1. Sắp xếp dữ liệu một lần: $O(n\log n)$.
2. Trả lời nhiều truy vấn ngưỡng bằng tìm kiếm nhị phân: mỗi truy vấn $O(\log n)$.

In [ ]:
def merge_sort(arr):
    if len(arr) <= 1:
        return arr[:]
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)


def merge(left, right):
    i = j = 0
    out = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    out.extend(left[i:])
    out.extend(right[j:])
    return out


def lower_bound(sorted_arr, target):
    lo, hi = 0, len(sorted_arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if sorted_arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

revenues = np.random.lognormal(mean=3.2, sigma=0.55, size=20).round(2).tolist()
sorted_revenues = merge_sort(revenues)
threshold = 30.0
pos = lower_bound(sorted_revenues, threshold)

print("Doanh thu ban đầu:", revenues)
print("Sau merge sort:", sorted_revenues)
print(f"Vị trí đầu tiên có doanh thu >= {threshold}:", pos)
print("Các giá trị đạt ngưỡng:", sorted_revenues[pos:])

In [ ]:
sizes = [200, 400, 800, 1600, 3200]
merge_times = []
builtin_times = []

for n in sizes:
    data = np.random.randint(0, 10_000, size=n).tolist()
    _, t_merge = time_function(merge_sort, data, repeats=3)
    _, t_builtin = time_function(sorted, data, repeats=3)
    merge_times.append(t_merge)
    builtin_times.append(t_builtin)

plt.figure(figsize=(8, 4))
plt.plot(sizes, merge_times, marker="o", label="Merge sort tự cài")
plt.plot(sizes, builtin_times, marker="o", label="sorted của Python")
plt.xlabel("Kích thước n")
plt.ylabel("Thời gian chạy trung vị (giây)")
plt.title("Đo thực nghiệm thời gian sắp xếp")
plt.legend()
plt.show()

pd.DataFrame({"n": sizes, "merge_sort": merge_times, "python_sorted": builtin_times})

### Nhận xét

Merge sort tự cài giúp hiểu rõ $O(n\log n)$, nhưng thư viện chuẩn của Python thường nhanh hơn nhiều vì được tối ưu ở mức thấp. Đây là lý do phân tích thuật toán cần đi cùng hiểu biết về cài đặt thực tế.

## 8. Giao thông vận tải: mạng tuyến và BFS

Ta mô hình hóa một mạng giao thông đô thị:

- Đỉnh là trạm hoặc điểm trung chuyển.
- Cạnh là tuyến đi trực tiếp giữa hai trạm.
- Nếu mỗi cạnh được xem như một chặng, BFS tìm đường đi **ít chặng nhất**.

Lưu ý: nếu mỗi đoạn có thời gian/chi phí khác nhau, BFS không còn tối ưu theo thời gian/chi phí. Khi đó cần thuật toán đường đi ngắn nhất có trọng số như Dijkstra, sẽ học ở bài sau.

In [ ]:
transport_edges = [
    ("BenXe", "ChoLon"),
    ("BenXe", "SanBay"),
    ("ChoLon", "Quan1"),
    ("ChoLon", "Quan5"),
    ("Quan1", "Quan3"),
    ("Quan3", "ThuDuc"),
    ("SanBay", "GoVap"),
    ("GoVap", "ThuDuc"),
    ("Quan5", "Quan8"),
    ("Quan8", "Quan1"),
]


def build_undirected_graph(edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    for u in graph:
        graph[u].sort()
    return dict(graph)

transport_graph = build_undirected_graph(transport_edges)
transport_graph

In [ ]:
def bfs(graph, source):
    visited = {source}
    dist = {source: 0}
    parent = {source: None}
    q = deque([source])
    order = []

    while q:
        u = q.popleft()
        order.append(u)
        for v in graph.get(u, []):
            if v not in visited:
                visited.add(v)
                dist[v] = dist[u] + 1
                parent[v] = u
                q.append(v)
    return order, dist, parent


def reconstruct_path(parent, target):
    if target not in parent:
        return None
    path = []
    cur = target
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1]

source, target = "BenXe", "ThuDuc"
order, dist, parent = bfs(transport_graph, source)
path = reconstruct_path(parent, target)

print("Thứ tự BFS:", order)
print("Khoảng cách theo số chặng:", dist)
print(f"Tuyến ít chặng từ {source} đến {target}:", " -> ".join(path))
print("Số chặng:", dist[target])

### Animation BFS

Animation dưới đây minh họa từng lớp BFS trên mạng giao thông. Màu cam là các đỉnh đã được phát hiện sau mỗi bước. Phần này thay cho video trong các lab AI/RL: sinh viên vẫn quan sát được diễn tiến của thuật toán.

In [ ]:
positions = {
    "BenXe": (0, 1),
    "ChoLon": (1, 1.6),
    "SanBay": (1, 0.4),
    "Quan5": (2, 2.1),
    "Quan8": (3, 1.7),
    "Quan1": (3, 1.0),
    "Quan3": (4, 1.2),
    "GoVap": (3, 0.2),
    "ThuDuc": (5, 0.7),
}

bfs_steps = order

fig, ax = plt.subplots(figsize=(8, 4.5))

def draw_graph(active_nodes):
    ax.clear()
    for u, v in transport_edges:
        x1, y1 = positions[u]
        x2, y2 = positions[v]
        ax.plot([x1, x2], [y1, y2], color="0.75", linewidth=2)
    for node, (x, y) in positions.items():
        color = "#f4a261" if node in active_nodes else "#d9e2ec"
        ax.scatter([x], [y], s=900, color=color, edgecolor="black", zorder=3)
        ax.text(x, y, node, ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.5, 5.6)
    ax.set_ylim(-0.2, 2.5)
    ax.set_title("Mô phỏng BFS trên mạng giao thông")
    ax.axis("off")


def update(frame):
    active = set(bfs_steps[:frame + 1])
    draw_graph(active)
    ax.text(0, -0.05, f"Bước {frame + 1}: đã phát hiện {', '.join(bfs_steps[:frame + 1])}", fontsize=10)

ani = animation.FuncAnimation(fig, update, frames=len(bfs_steps), interval=900, repeat=False)
plt.close(fig)
display(HTML(ani.to_jshtml()))

## 9. DFS: kiểm tra kết nối và thành phần liên thông

Trong vận tải, một câu hỏi thực tế là: mạng có bị tách cụm không? Nếu một khu vực bị cô lập, hệ thống điều phối cần phát hiện sớm.

DFS hoặc BFS đều tìm được thành phần liên thông trong đồ thị vô hướng với độ phức tạp $O(|V|+|E|)$.

In [ ]:
def dfs_component(graph, start, visited):
    stack = [start]
    component = []
    visited.add(start)
    while stack:
        u = stack.pop()
        component.append(u)
        for v in graph.get(u, []):
            if v not in visited:
                visited.add(v)
                stack.append(v)
    return component


def connected_components(graph):
    visited = set()
    comps = []
    for node in sorted(graph):
        if node not in visited:
            comps.append(sorted(dfs_component(graph, node, visited)))
    return comps

incident_edges = transport_edges + [("KhoHang", "Cang"), ("Cang", "KhuCN")]
incident_graph = build_undirected_graph(incident_edges)
components = connected_components(incident_graph)

print("Số thành phần liên thông:", len(components))
for i, comp in enumerate(components, 1):
    print(f"Thành phần {i}:", comp)

## 10. Trí tuệ nhân tạo: chọn mẫu gán nhãn bằng tham lam và lan truyền nhãn bằng BFS

Trong một bài toán AI, nhãn dữ liệu thường đắt. Ta mô phỏng tập điểm 2D gồm hai nhóm. Ý tưởng thực hành:

1. Dùng chiến lược tham lam để chọn một số điểm đại diện cần gán nhãn.
2. Xây dựng đồ thị tương đồng: hai điểm nối cạnh nếu đủ gần nhau.
3. Dùng BFS để lan truyền nhãn từ điểm đã gán nhãn đến các điểm chưa nhãn trong cùng cụm gần.

Đây là mô hình đơn giản của **semi-supervised learning** dựa trên đồ thị.

In [ ]:
def make_cluster(center, n, scale=0.35):
    return np.random.normal(loc=center, scale=scale, size=(n, 2))

X0 = make_cluster(center=(-1.2, 0.0), n=18)
X1 = make_cluster(center=(1.2, 0.2), n=18)
X = np.vstack([X0, X1])
true_labels = np.array([0] * len(X0) + [1] * len(X1))

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=true_labels, cmap="coolwarm", edgecolor="black")
plt.title("Dữ liệu AI mô phỏng: hai cụm điểm")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.show()

In [ ]:
def euclidean(a, b):
    return float(np.linalg.norm(a - b))


def greedy_farthest_first(points, k):
    center = points.mean(axis=0)
    first = int(np.argmin([euclidean(p, center) for p in points]))
    chosen = [first]

    while len(chosen) < k:
        best_idx = None
        best_dist = -1
        for i, p in enumerate(points):
            if i in chosen:
                continue
            d = min(euclidean(p, points[j]) for j in chosen)
            if d > best_dist:
                best_dist = d
                best_idx = i
        chosen.append(best_idx)
    return chosen

selected = greedy_farthest_first(X, k=4)
print("Các điểm được chọn để gán nhãn:", selected)
print("Nhãn thật của các điểm được chọn:", true_labels[selected].tolist())

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c="lightgray", edgecolor="black", label="Chưa gán nhãn")
plt.scatter(X[selected, 0], X[selected, 1], c=true_labels[selected], cmap="coolwarm", s=180, edgecolor="black", marker="*", label="Chọn gán nhãn")
plt.title("Chọn mẫu đại diện bằng tham lam")
plt.legend()
plt.show()

In [ ]:
def build_similarity_graph(points, radius):
    graph = defaultdict(list)
    n = len(points)
    for i in range(n):
        graph[i]
    for i in range(n):
        for j in range(i + 1, n):
            if euclidean(points[i], points[j]) <= radius:
                graph[i].append(j)
                graph[j].append(i)
    return {u: sorted(vs) for u, vs in graph.items()}


def propagate_labels(graph, seed_indices, seed_labels):
    predicted = {idx: label for idx, label in zip(seed_indices, seed_labels)}
    q = deque(seed_indices)

    while q:
        u = q.popleft()
        for v in graph[u]:
            if v not in predicted:
                predicted[v] = predicted[u]
                q.append(v)
    return predicted

radius = 0.75
sim_graph = build_similarity_graph(X, radius=radius)
predicted_map = propagate_labels(sim_graph, selected, true_labels[selected])
predicted_labels = np.array([predicted_map.get(i, -1) for i in range(len(X))])
coverage = np.mean(predicted_labels != -1)
accuracy_on_covered = np.mean(predicted_labels[predicted_labels != -1] == true_labels[predicted_labels != -1])

print("Bán kính nối cạnh:", radius)
print("Tỷ lệ được lan truyền nhãn:", round(float(coverage), 3))
print("Độ chính xác trên điểm đã được gán nhãn:", round(float(accuracy_on_covered), 3))

In [ ]:
plt.figure(figsize=(6, 5))
for u, vs in sim_graph.items():
    for v in vs:
        if u < v:
            plt.plot([X[u, 0], X[v, 0]], [X[u, 1], X[v, 1]], color="0.85", linewidth=1)

colors = ["tab:blue" if y == 0 else "tab:red" if y == 1 else "white" for y in predicted_labels]
plt.scatter(X[:, 0], X[:, 1], c=colors, edgecolor="black", s=70)
plt.scatter(X[selected, 0], X[selected, 1], c=true_labels[selected], cmap="coolwarm", s=220, edgecolor="black", marker="*")
plt.title("Lan truyền nhãn trên đồ thị tương đồng")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.show()

### Thí nghiệm nhỏ: ảnh hưởng của bán kính nối cạnh

Nếu bán kính quá nhỏ, nhiều điểm bị cô lập và không nhận được nhãn. Nếu bán kính quá lớn, hai cụm có thể nối lẫn nhau và lan truyền nhãn sai. Đây là ví dụ cho thấy mô hình đồ thị quyết định chất lượng thuật toán.

In [ ]:
radii = np.linspace(0.25, 1.4, 10)
coverages = []
accuracies = []

for r in radii:
    g = build_similarity_graph(X, radius=float(r))
    pred_map = propagate_labels(g, selected, true_labels[selected])
    pred = np.array([pred_map.get(i, -1) for i in range(len(X))])
    covered = pred != -1
    coverages.append(np.mean(covered))
    accuracies.append(np.mean(pred[covered] == true_labels[covered]) if covered.any() else 0)

plt.figure(figsize=(8, 4))
plt.plot(radii, coverages, marker="o", label="Tỷ lệ có nhãn")
plt.plot(radii, accuracies, marker="o", label="Độ chính xác trên điểm có nhãn")
plt.xlabel("Bán kính nối cạnh")
plt.ylabel("Tỷ lệ")
plt.title("Ảnh hưởng của mô hình đồ thị trong lan truyền nhãn")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

pd.DataFrame({"radius": radii.round(2), "coverage": np.round(coverages, 3), "accuracy": np.round(accuracies, 3)})

## 11. Kiểm thử trường hợp biên

Một lab thuật toán không nên chỉ chạy trên dữ liệu đẹp. Các trường hợp nhỏ và sát biên giúp phát hiện lỗi cài đặt.

In [ ]:
def run_basic_tests():
    assert merge_sort([]) == []
    assert merge_sort([3, 1, 2]) == [1, 2, 3]
    assert lower_bound([1, 3, 5], 4) == 2
    assert lower_bound([1, 3, 5], 6) == 3

    tiny = pd.DataFrame({
        "campaign": ["X", "Y"],
        "cost": [5, 6],
        "benefit": [10, 13],
        "risk": [1, 1],
    })
    tiny["ratio"] = tiny["benefit"] / tiny["cost"]
    assert dp_campaign_selection(tiny, 5)[:3] == (["X"], 5, 10)

    g = build_undirected_graph([("A", "B"), ("C", "D")])
    comps = connected_components(g)
    assert len(comps) == 2

    order, dist, parent = bfs({"A": ["B"], "B": ["A"]}, "A")
    assert reconstruct_path(parent, "B") == ["A", "B"]
    print("Tất cả kiểm thử cơ bản đã qua.")

run_basic_tests()

## 12. Tổng kết

Qua ba nhóm tình huống, ta thấy cùng một thuật toán có thể xuất hiện dưới nhiều hình thức khác nhau:

- **Độ phức tạp** giúp dự đoán khi dữ liệu tăng từ vài chục lên vài nghìn hoặc vài triệu phần tử.
- **Tham lam** phù hợp khi tiêu chí cục bộ có ý nghĩa rõ ràng, nhưng cần kiểm tra phản ví dụ.
- **Chia để trị** là nền tảng của sắp xếp/tìm kiếm hiệu quả.
- **Quy hoạch động** hữu ích khi phải thử nhiều lựa chọn và có ràng buộc ngân sách/tài nguyên.
- **BFS/DFS** biến các bài toán kết nối, lan truyền, truy vết thành bài toán đồ thị.

Trong bài sau, khi cạnh có trọng số, bài toán giao thông sẽ cần thuật toán đường đi ngắn nhất thay vì BFS.

## 13. Bài tập

1. Thay đổi ngân sách marketing từ 30 thành 25, 35 và 40. So sánh nghiệm tham lam với nghiệm quy hoạch động.
2. Thêm một chiến dịch mới có chi phí thấp nhưng lợi ích rất cao. Quan sát xem greedy và DP thay đổi thế nào.
3. Thay hàm chọn mẫu AI bằng chọn ngẫu nhiên 4 điểm. Chạy nhiều lần và so sánh độ phủ/độ chính xác trung bình với chiến lược farthest-first.
4. Trong mạng giao thông, thêm một cạnh trực tiếp từ SanBay đến ThuDuc. BFS trả về đường đi nào? Vì sao?
5. Tạo một mạng giao thông có hai thành phần liên thông và viết cảnh báo nếu kho hàng nằm trong thành phần khác với cảng.
6. Viết báo cáo ngắn 5-7 dòng: với mỗi miền kinh tế, giao thông, AI, nêu một giả định mô hình có thể làm kết quả thuật toán khác thực tế.

## 14. Câu hỏi thảo luận

1. Vì sao ba lô 0-1 không nên giải bằng tham lam nếu cần nghiệm tối ưu?
2. Khi nào BFS không còn phù hợp để tìm đường đi tốt nhất trong giao thông?
3. Tại sao đồ thị tương đồng trong AI nhạy với tham số bán kính?
4. Nếu dữ liệu tăng lên 1 triệu điểm, phần nào của notebook sẽ trở thành nút thắt?
5. Trong thực tế, ngoài độ phức tạp, còn những yếu tố nào ảnh hưởng đến việc chọn thuật toán?

## Giải bài tập

### 1. Thay đổi ngân sách marketing và so sánh tham lam với quy hoạch động

In [1]:
print_section("Bài tập 1: Thay đổi ngân sách marketing")

budgets_to_test = [25, 35, 40]

results_data = []

for b in budgets_to_test:
    greedy_res = greedy_campaign_selection(campaigns, b)
    dp_res = dp_campaign_selection(campaigns, b)

    print(f"\n--- Ngân sách: {b} ---")
    print("Tham lam:", greedy_res)
    print("Quy hoạch động:", dp_res[:3])

    results_data.append({
        "Budget": b,
        "Method": "Greedy",
        "Chosen Campaigns": greedy_res[0],
        "Total Cost": greedy_res[1],
        "Total Benefit": greedy_res[2]
    })
    results_data.append({
        "Budget": b,
        "Method": "Dynamic Programming",
        "Chosen Campaigns": dp_res[0],
        "Total Cost": dp_res[1],
        "Total Benefit": dp_res[2]
    })

display(pd.DataFrame(results_data))

NameError: name 'print_section' is not defined

#### Nhận xét bài tập 1:
Từ bảng kết quả, chúng ta có thể thấy rằng:
- Với ngân sách 25, cả hai phương pháp đều chọn được các chiến dịch giống nhau và tổng lợi ích/chi phí tương đương.
- Với ngân sách 35 và 40, phương pháp quy hoạch động cho thấy tổng lợi ích cao hơn phương pháp tham lam, mặc dù đôi khi tổng chi phí có thể khác nhau do cách tiếp cận khác nhau trong việc chọn các chiến dịch. Điều này minh họa rằng thuật toán tham lam có thể không cho kết quả tối ưu toàn cục.